# Interpretabilidade

Saber que o modelo acerta X% é útil. Saber *por que* ele decide o que decide é o que importa pra quem vai usar na prática. Se a gravadora recebe um "baixo potencial" e não sabe o motivo, a predição não serve de nada.

In [ ]:
import pandas as pd
import numpy as np
import shap
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/tracks_clean.csv")

FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "duration_ms",
]

X = df[FEATURES]
y = df["is_hit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

xgb = XGBClassifier(
    n_estimators=200, max_depth=6,
    learning_rate=0.1, eval_metric="logloss", random_state=42,
)
xgb.fit(X_train, y_train)
print("Modelo treinado.")

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
print(f"SHAP calculado para {len(X_test)} amostras.")

## Importância global das features

Pra cada feature, quanto ela pesa em média nas decisões do modelo. Features com barra maior são as que mais empurram a predição pra um lado ou pro outro.

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "mean_shap": mean_abs_shap,
}).sort_values("mean_shap", ascending=True)

fig = px.bar(
    importance_df, x="mean_shap", y="feature",
    orientation="h",
    title="Importância das features (SHAP médio absoluto)",
    labels={"mean_shap": "Impacto médio na predição", "feature": ""},
)
fig.update_layout(height=400, template="plotly_white")
fig.show()

## Beeswarm

Cada ponto é uma música. A posição horizontal mostra se a feature empurrou a predição pra hit (direita) ou não-hit (esquerda). A cor mostra o valor real da feature naquela música.

In [ ]:
shap_df = pd.DataFrame(shap_values, columns=FEATURES)
feature_order = importance_df["feature"].tolist()

sample_size = min(500, len(shap_df))
sample_idx = np.random.RandomState(42).choice(len(shap_df), sample_size, replace=False)

fig = go.Figure()
for i, feat in enumerate(feature_order):
    feat_vals = X_test.iloc[sample_idx][feat].values
    normalized = (feat_vals - feat_vals.min()) / (feat_vals.max() - feat_vals.min() + 1e-10)

    fig.add_trace(go.Scatter(
        x=shap_df.iloc[sample_idx][feat].values,
        y=np.random.normal(i, 0.15, size=sample_size),
        mode="markers",
        marker=dict(
            size=3, color=normalized,
            colorscale="RdBu_r", opacity=0.6,
        ),
        name=feat, showlegend=False,
        hovertemplate=f"{feat}<br>SHAP: %{{x:.3f}}<extra></extra>",
    ))

fig.update_layout(
    title="SHAP Beeswarm (amostra de 500 músicas)",
    xaxis_title="Valor SHAP (impacto na predição)",
    yaxis=dict(tickvals=list(range(len(feature_order))), ticktext=feature_order),
    height=500, template="plotly_white",
)
fig.show()

## Waterfall: um hit classificado corretamente

Pego uma música que é hit e o modelo acertou. O waterfall mostra quais features empurraram a predição pra cima (hit) e quais tentaram puxar pra baixo.

In [ ]:
y_pred = xgb.predict(X_test)

correct_hits = X_test[(y_test == 1) & (y_pred == 1)]
if len(correct_hits) > 0:
    idx = correct_hits.index[0]
    pos = X_test.index.get_loc(idx)
    sv = shap_values[pos]

    sorted_idx = np.argsort(np.abs(sv))
    sorted_feats = [FEATURES[i] for i in sorted_idx]
    sorted_vals = sv[sorted_idx]

    fig = go.Figure(go.Waterfall(
        y=sorted_feats, x=sorted_vals,
        orientation="h",
        connector={"line": {"color": "gray"}},
    ))
    fig.update_layout(
        title="Waterfall SHAP — hit classificado corretamente",
        height=400, template="plotly_white",
    )
    fig.show()

    song = df.loc[idx]
    print(f"Música: {song['name']} — {song['artist']}")
    print(f"Popularity: {song['popularity']}")

## Waterfall: um não-hit classificado corretamente

Agora o oposto: uma música que não é hit e o modelo acertou. Quais features sinalizaram que ela não tinha perfil de hit?

In [ ]:
correct_nonhits = X_test[(y_test == 0) & (y_pred == 0)]
if len(correct_nonhits) > 0:
    idx = correct_nonhits.index[0]
    pos = X_test.index.get_loc(idx)
    sv = shap_values[pos]

    sorted_idx = np.argsort(np.abs(sv))
    sorted_feats = [FEATURES[i] for i in sorted_idx]
    sorted_vals = sv[sorted_idx]

    fig = go.Figure(go.Waterfall(
        y=sorted_feats, x=sorted_vals,
        orientation="h",
        connector={"line": {"color": "gray"}},
    ))
    fig.update_layout(
        title="Waterfall SHAP — não-hit classificado corretamente",
        height=400, template="plotly_white",
    )
    fig.show()

    song = df.loc[idx]
    print(f"Música: {song['name']} — {song['artist']}")
    print(f"Popularity: {song['popularity']}")

## Resposta à pergunta de negócio

**"Quais características de áudio um single precisa ter pra ter maior chance de sucesso?"**

Baseado no que o modelo aprendeu com 80 mil músicas:

- **Danceability alta** — músicas dançáveis performam melhor. Se o single não convida o corpo a se mexer, as chances caem.
- **Loudness dentro do padrão de mercado** — produção com volume competitivo (perto de -5 a -7 dB) é praticamente requisito. Faixas com loudness muito baixo ficam pra trás.
- **Energy alta, mas sem exagero** — ter energia importa. Mas as faixas que mais estouram ficam num patamar alto sem serem agressivas.
- **Duração curta a moderada** — o sweet spot fica entre 2:30 e 3:30. Músicas longas demais perdem ouvintes no streaming.
- **Acousticness baixa** — produção eletrônica domina os charts. Faixas puramente acústicas têm menos chance.
- **Valência e tempo importam pouco** — não faz muita diferença se a música é triste ou alegre, lenta ou rápida. O que conta é a produção e o quão dançável ela é.